# Lists, dicts, and mutation

What it means for two names to share one thing, and how to stop sharing it on purpose.

## How we'll work today

As before: **Predict** before you run. Write your answer down, compare with your neighbour, then run the cell.

## Before we start: `import`

Every exercise file starts with a line like `from puzzle_input import TRANSCRIPT`, and every test file with `from ex03 import ...`. `import` makes names from somewhere else available in your file:

- A **module** is a single Python file. `puzzle_input.py` is the module `puzzle_input`; your `ex03.py` is the module `ex03`.
- A **package** is a folder of modules that belong together, imported the same way. Parts of it are reached with a dot: `import urllib.request` imports the module `request` from the package `urllib`.

When you import something, the difference rarely matters: `import x` looks the same either way.

Where they come from:

- Your own files, like `puzzle_input`.
- The **standard library**: modules and packages that come with every Python installation. Today we'll use a few of them. `math` is one: it holds mathematical functions and constants.
- Packages someone else wrote that you install, like `pytest` (and later `pandas`). `uv` installs them for you, listed in `pyproject.toml`.

In [ ]:
import math

print(math.sqrt(16))
print(math.pi)

`import math` gives you one new name, `math`; everything inside is reached with a dot, like a method. **Predict:** what happens here?

In [ ]:
import math

print(sqrt(16))

`NameError`: `import math` only defined `math`, not `sqrt`. To get a single name directly, use `from ... import ...`, as the exercise files do:

In [ ]:
from math import sqrt

print(sqrt(16))

Imports go at the **top** of a file, so a reader sees at a glance what the file depends on. (In this notebook they sit next to where they're first used, to keep the examples together.)

# Part 1: Mutation vs. references, revisited

## Recap: `b = a`

On day 1 we saw `b = a; a = 2`: reassigning `a` doesn't touch `b`, because `a = 2` makes `a` point at a new value. Now we don't reassign, we call a method on the list.

**Predict:** what does this print?

In [ ]:
a = [1, 2, 3]
b = a
b.append(4)
print(a)
print(b)

`.append(...)` doesn't make a new list, it changes the one that's already there. `a` and `b` are two labels on **the same** list: there is one list object, and changing it through either label shows up through both. *Watch it on pythontutor.com: one box, two arrows.*

`a is b` asks "are these the same object?"; `a == b` asks "are their contents equal?". 

**Predict:** what do these print?

In [ ]:
print(a is b)
print(a == b)
c = [1, 2, 3, 4]
print(a == c, a is c)

## Back to Exercise 2

`run_with(program, noun, verb)` puts `noun` and `verb` into `program` and runs it. Here is a stand-in that does the same kind of thing, without the computer.

**Predict:** what does the second `print` show?

In [ ]:
def run_with_demo(memory, noun, verb):
    memory[1] = noun
    memory[2] = verb
    memory[0] = 99  # pretend this is what running the program does
    return memory[0]


program = [1, 0, 0, 0]
print(run_with_demo(program, 5, 6))
print(program)

`run_with_demo` writes into the list it was given — and `run` does too, that's the point of the computer. So every call with the **same** `program` list changes it permanently.

`find_noun_verb` calls `run_with` up to 10000 times, trying different nouns and verbs on that one list. After the first call, `program` isn't the original program anymore — it's whatever the first run left behind. Every later call runs a different program than the one you think you're running.

## Copying a list

`.copy()` gives you a **new** list with the same elements, so the original stays untouched:

In [ ]:
original = [1, 2, 3]
copy = original.copy()

copy.append(4)
print(original)
print(copy)

**Predict:** which of these make a copy, and which just add another label?

In [ ]:
x = [1, 2, 3]
y = x
z = x.copy()
w = x[:]
u = list(x)

x.append(99)
print("x:", x)
print("y:", y)
print("z:", z)
print("w:", w)
print("u:", u)

`y` and `v` are the same object as `x` (`y is x` is `True`); `z`, `w` and `u` are new lists. A slice like `x[:]` always builds a new list, so `x[:]` (from start to end) is a copy too, and so is `list(x)`, which builds a new list from anything you can loop over. All three work; we'll use `.copy()` from here on, because it says what you mean.

The fix for Exercise 2: `run_with` (or whoever calls `run`) copies the list **before** changing it, e.g. `memory = program.copy()` as the first line.

## Lists inside lists

**Predict:** what does `original` look like after this?

In [ ]:
original = [[1, 2], [3, 4]]
shallow = original.copy()
shallow[0].append(99)
print(original)

Copying doesn't copy what's inside: `.copy()` makes a **new outer list**, but the elements inside (here, two inner lists) are still the *same* objects, just with new labels pointing at them. `shallow[0]` and `original[0]` are the same list. This is called a **shallow copy**. It matters for lists of lists; it won't matter for Exercise 3's dicts, where every value is a plain number — numbers can't be mutated at all, so there's nothing to share.

If you do need the inner lists copied too, the standard library's `copy` module has `deepcopy`, which copies all the way down:

In [ ]:
from copy import deepcopy

original = [[1, 2], [3, 4]]
deep = deepcopy(original)
deep[0].append(99)
print(original)
print(deep)

## A list as a default value

In week 2 we gave parameters default values, e.g. `def greet(name, greeting="Hello")`. **Predict:** what do the two `print`s show?

In [ ]:
def add_item(item, items=[]):
    items.append(item)
    return items


print(add_item("a"))
print(add_item("b"))

The default value is created **once**, when `def` runs, not every time the function is called. Every call that leaves out `items` gets the **same** list, and `.append` changes that one list, so it keeps growing from call to call. It's the `b = a` problem again, hidden inside a function definition.

The usual fix: use `None` as the default and make a new list inside the function.

In [ ]:
def add_item(item, items=None):
    if items is None:
        items = []
    items.append(item)
    return items


print(add_item("a"))
print(add_item("b"))

Numbers, strings and `None` are safe defaults: they can't be changed, so sharing them does no harm. Lists and dicts as defaults are almost always a bug.

# Part 2: Dictionaries

## A dict maps keys to values

**Predict:** what does each line print?

In [ ]:
ages = {"Ada": 36, "Grace": 34}
print(ages["Ada"])
ages["Alan"] = 41
print(ages)
print(len(ages))

- `{key: value, ...}` creates a dict.
- `d[key]` reads the value for `key`; `d[key] = value` sets it (adding the key if it's new, overwriting it if it already exists).
- Keys are unique: setting an existing key replaces its value, it doesn't add a second entry.

**Predict:** what happens here?

In [ ]:
ages = {"Ada": 36}
print(ages["Grace"])

`KeyError`, Python's way of saying "that key isn't there" — the dict equivalent of an `IndexError`. Two ways to ask without risking that:

In [ ]:
ages = {"Ada": 36}
print("Grace" in ages)
print(ages.get("Grace"))
print(ages.get("Grace", 0))

`in` asks whether a key exists; `.get(key, default)` reads the value, or `default` (`None` if you don't give one) if the key is missing. `.get` never raises.

## Going through a dict

**Predict:** what does each loop print, and in which order?

In [ ]:
ages = {"Ada": 36, "Grace": 34, "Alan": 41}

for name in ages:
    print(name)

for name, age in ages.items():
    print(name, age)

for age in ages.values():
    print(age)

Looping over a dict directly gives you its **keys**, in the order they were added. `.items()` gives `(key, value)` pairs, unpacked the same way `enumerate(...)` pairs are. `.keys()` and `.values()` also exist, mostly useful on their own (`sorted(ages.values())`, `"Ada" in ages.keys()`, though plain `in ages` already does the second one).

## The accumulator pattern, for dicts

Counting or summing *per key* is exactly Exercise 1's `floor` pattern (a name set before the loop, updated inside it), just with a dict instead of a single number. **Predict:** what does this print?

In [ ]:
words = ["to", "be", "or", "not", "to", "be"]
counts = {}
for word in words:
    counts[word] = counts.get(word, 0) + 1
print(counts)

`counts.get(word, 0)` reads the count so far, or `0` for a word seen for the first time, then `+ 1` and the result is written back. Without the `, 0)`, the first time any word shows up would raise a `KeyError`.

Exercise 3's `directory_sizes` does the same thing: for every file, add its size to **every** directory it's inside — the dict key is a directory path instead of a word.

Counting is common enough that Python's **standard library** already has it: `Counter` from the `collections` module. Before you write a general-purpose helper yourself, check whether it already exists.

In [ ]:
from collections import Counter

print(Counter(words))
print(Counter(words).most_common(1))

`Counter` only counts. For summing or anything else per key, `defaultdict` from the same module fills in a starting value for a key it hasn't seen yet — `int()` is `0` — so you don't need `.get(key, 0)`:

In [ ]:
from collections import defaultdict

counts = defaultdict(int)
for word in words:
    counts[word] += 1
print(counts)
print(dict(counts))

## `b = a` with a dict

**Predict:** what does this print?

In [ ]:
a = {"x": 1}
b = a
b["y"] = 2
print(a)

Same rule as lists: dicts are mutable, and `b = a` is a second label, not a second dict. Copy a dict with `.copy()` (or `dict(d)`) when you need to change one without touching the other. (`d[:]` doesn't work for dicts; slicing is a list/string/sequence thing.)

In [ ]:
a = {"x": 1}
b = a.copy()
b["y"] = 2
print(a)
print(b)

# Part 3: Comprehensions

## Building a list from another iterable

**Predict:** what does this print?

In [ ]:
numbers = [1, 2, 3, 4, 5]
squares = [n * n for n in numbers]
print(squares)

`[<expression> for <name> in <iterable>]` is a **list comprehension**: it's the same thing as

In [ ]:
squares = []
for n in numbers:
    squares.append(n * n)
print(squares)

written on one line. Use it when the loop's only job is to build a list by appending one transformed value per element — not when it does something else too (printing, branching into very different cases, updating several names).

## With a condition

**Predict:** what's in `even_squares`?

In [ ]:
even_squares = [n * n for n in numbers if n % 2 == 0]
print(even_squares)

`if <condition>` after the `for` keeps only the elements that pass it, *before* the expression is applied. With `sum(...)` wrapped around it instead of `[...]`, it adds up only the values that pass — you'll need this shape in Exercise 3:

In [ ]:
ages = {"Ada": 36, "Grace": 34, "Alan": 41}
total = sum(age for age in ages.values() if age >= 36)
print(total)

Round brackets instead of square ones, with no list in between: a **generator expression**. `sum(...)` (or `max`, `any`, `list`, ...) consumes the values one at a time, so no list is ever built just to be thrown away.

## Dict comprehensions

**Predict:** what does this print?

In [ ]:
ages = {"Ada": 36, "Grace": 34, "Alan": 41}
adults_only = {name: age for name, age in ages.items() if age >= 36}
print(adults_only)

`{<key>: <value> for ... in ... if ...}` builds a **new** dict — the original `ages` is untouched. So to get a dict *without* one key, keep every pair except that one:

In [ ]:
def without_key(d, key):
    return {k: v for k, v in d.items() if k != key}


result = without_key(ages, "Ada")
print(result)
print(ages)  # unchanged

Another way to write it, using `del`: `del d[key]` removes `key` (and its value) from `d`; like `d[key]`, it raises `KeyError` if the key isn't there.

**Predict:** what do the two `print`s show?

In [ ]:
def without_key_del(d, key):
    new_d = d
    del new_d[key]
    return new_d


ages = {"Ada": 36, "Grace": 34}
result = without_key_del(ages, "Ada")
print(result)
print(ages)

Same bug as Part 1, now with a dict: `new_d = d` doesn't copy, so `del new_d[key]` deletes it from the one and only dict both names point to. The comprehension version above never had this problem, because it builds a genuinely new dict. Copying first fixes this version too: `new_d = d.copy()`, then `del new_d[key]`.

Exercise 3's `delete_directory` has the same rule: it returns a new dict, and the one it was given must not change.

# Part 4: Before Exercise 3

## A list as a stack

`.append(x)` adds `x` at the end of a list; `.pop()` removes the **last** element and returns it. Used together, the end of the list is the top of a pile: last in, first out. **Predict:** what do the three `print`s show?

In [ ]:
current = []
current.append("photos")
current.append("summer")
print(current)
print(current.pop())
print(current)

**Predict:** what's in `seen`?

In [ ]:
current = ["photos"]
seen = []
seen.append(current)
current.append("summer")
print(seen)

`.append` and `.pop` change the list in place, so everything from Part 1 applies: `seen` holds a second label on `current`, not a snapshot of what it looked like at the time. To keep what a list looks like *now*, store a copy, or something built from it, like a string.

## Summary

- `b = a` (list or dict) makes a **second label on the same object**; changing it through either name is visible through both
- `a is b` asks "same object?"; `a == b` asks "equal contents?"
- Copy a list or a dict with `.copy()` (`list(...)`/`dict(...)` and a slice `[:]` also copy)
- A shallow copy makes a new outer container, but shares whatever's *inside* it — harmless when the values are plain numbers or strings
- `copy.deepcopy(...)` copies nested lists all the way down
- A default value is created once, when `def` runs: never use a list or dict as a default, use `None` and create it inside the function
- `.append(x)` and `.pop()` use a list as a stack; both change it in place
- `d[key]` raises `KeyError` for a missing key; `d.get(key, default)` doesn't
- `d.get(key, 0) + 1`, written back to `d[key]`: the accumulator pattern for dicts
- `collections.Counter(...)` counts for you, `collections.defaultdict(int)` starts every new key at `0`: check the standard library before writing a helper
- `for key, value in d.items():` to go through both at once
- A list/dict comprehension replaces a loop whose only job is building a new list/dict
- `sum(x for x in ... if ...)`: a generator expression, no throwaway list built
- A comprehension always builds something **new** — it's the safe way to "change" a dict without mutating the original